## TARGETS

In [ ]:
# REGEX Script for predicting whether cve targetting protocol, firmware, driver, library, service or software
import pandas as pd, re
from pathlib import Path

inp_path = "cve.csv"
assert Path(inp_path).exists(), f"{inp_path} not found"

df = pd.read_csv(inp_path, dtype=str).fillna("")

# Normalize column names
if "cve_id" not in df.columns:
    possible = [c for c in df.columns if c.lower().startswith("cve")]
    if possible:
        df = df.rename(columns={possible[0]: "cve_id"})
    else:
        raise ValueError("No cve_id column found in input CSV")

if "description_en" not in df.columns:
    possible = [c for c in df.columns if "description" in c.lower()]
    if possible:
        df = df.rename(columns={possible[0]: "description_en"})
    else:
        raise ValueError("No description_en column found in input CSV")

text_series = df["description_en"].astype(str)

# patterns
import re
protocol_pat = re.compile(r"\b(protocol|modbus|opc ua|opc|http|https|tcp|udp|mqtt|ssh|rdp|ftp|smtp|dnp3|s7comm|ethernet/ip|coap)\b", re.I)
firmware_pat = re.compile(r"\b(firmware|bios|uefi|embedded|boot|firm-?ware|firmwares)\b", re.I)
driver_pat = re.compile(r"\b(driver|device driver|kernel module|nvme|gpu driver|win32k)\b", re.I)
library_pat = re.compile(r"\b(lib|library|openssl|glibc|jar|dll|\.so\b|package)\b", re.I)
service_pat = re.compile(r"\b(service|daemon|server|web server|mail server|iis|nginx|apache|daemon)\b", re.I)
software_pat = re.compile(r"\b(application|software|client|browser|plugin|extension|component|app)\b", re.I)

# compute boolean arrays (vectorized, using str.contains avoids Python loop)
is_protocol = text_series.str.contains(protocol_pat, regex=True, na=False)
is_firmware = text_series.str.contains(firmware_pat, regex=True, na=False)
is_driver = text_series.str.contains(driver_pat, regex=True, na=False)
is_library = text_series.str.contains(library_pat, regex=True, na=False)
is_service = text_series.str.contains(service_pat, regex=True, na=False)
is_software = text_series.str.contains(software_pat, regex=True, na=False)

# build results strings
n = len(df)
results = []
for i in range(n):
    tags = []
    if is_software.iat[i]:
        tags.append("software")
    if is_service.iat[i]:
        tags.append("service")
    if is_library.iat[i]:
        tags.append("library")
    if is_driver.iat[i]:
        tags.append("driver")
    if is_firmware.iat[i]:
        tags.append("firmware")
    if is_protocol.iat[i]:
        tags.append("protocol")
    if not tags:
        tags = ["software"]
    results.append(tags)

out_df = pd.DataFrame({"cve_id": df["cve_id"], "result": results})
out_path = "cve_classified_target.csv"
out_df.to_csv(out_path, index=False)


## I5_Asset

In [ ]:
## Code to predict where each product is located in I5 Asset
import os, re, pandas as pd

RAW = "CISA_ICS_ADV_Master.csv"
OUT  = "ics_master_i5asset.csv"

def find_col(df, candidates):
    norm = {c.lower().replace(" ", "_"): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "_")
        if key in norm: return norm[key]
    for c in df.columns:
        k = c.lower().replace(" ", "_")
        if "product" in k and "affect" in k:
            return c
    raise KeyError("Product_Affected-like column not found")

raw = pd.read_csv(RAW, dtype=str)
ics_col = "ICS-CERT_Number" if "ICS-CERT_Number" in raw.columns else find_col(raw, ["ICS-CERT_Number","ICSCERT_Number","ICS Number"])
prod_aff_col = find_col(raw, ["Product_Affected","Products_Affected","Product Affected","Products Affected"])
raw_subset = raw[[ics_col, prod_aff_col]].rename(columns={ics_col:"ICS-CERT_Number", prod_aff_col:"Product_Affected"})
raw_subset["Product_Affected"] = raw_subset["Product_Affected"].fillna("").astype(str)

agg = (raw_subset.groupby("ICS-CERT_Number", as_index=False)
       .agg({"Product_Affected": lambda s: " | ".join(sorted(set([x.strip() for x in s if str(x).strip()])))}))

patterns = {
    "PLC/Controller": [r"\bplc\b", r"\b(plc|controller)\s*cpu\b", r"\bs7[-\s]?\d+\b", r"\bmodicon\b", r"\bm580\b", r"\bm340\b",
                       r"\bcontrollogix\b", r"\bcompactlogix\b", r"\bomron\b", r"\bnx\d*\b", r"\bnj\d*\b", r"\bq-?series\b",
                       r"\bfx\d*u?\b", r"\bmelsec\b", r"\bpac\b", r"\bmicrologix\b", r"\bs7-1200\b", r"\bs7-1500\b", r"\bcodesys\b"],
    "HMI/SCADA": [r"\bscada\b", r"\bhmi\b", r"\bwincc\b", r"\bwincc\s*(oa|open architecture)\b", r"\bfactorytalk\s*(view|se|me)\b",
                  r"\bpanelview\b", r"\bwonderware\b", r"\bintouch\b", r"\bigntion\b", r"\bifix\b", r"\bzenon\b", r"\bmovicon\b",
                  r"\b(cimplicity|ifix)\b"],
    "Engineering Workstation": [r"\btia\s*portal\b", r"\bstep\s*7\b", r"\bstudio\s*5000\b", r"\b(unity\s*pro|control\s*expert)\b",
                                r"\bgx\s*works\b", r"\bcx[-\s]?one\b", r"\bprogrammer\b", r"\bengineering\s*(workstation|station|software)\b",
                                r"\bconfiguration\s*tool\b"],
    "Historian/Database": [r"\bhistorian\b", r"\bpi\s*system\b", r"\bproficy\s*historian\b", r"\bip21\b"],
    "OPC Server/Middleware": [r"\bopc(\s*ua|\s*da)?\b", r"\bkepware\b", r"\bmatrikon\b", r"\bopc\s*server\b"],
    "Industrial Network (Switch/Router/Firewall)": [r"\bscalance\b", r"\bsinema\b", r"\bstratix\b", r"\bcisco\s*ie\b",
                                                    r"\bruggedcom\b", r"\bwestermo\b", r"\bmoxa\b",
                                                    r"\bindustrial\s*(switch|router|firewall)\b", r"\bswitch\b", r"\brouter\b", r"\bfirewall\b"],
    "Industrial Wireless/AP": [r"\biwlan\b", r"\baccess\s*point\b", r"\bwireless\b", r"\b802\.11\b", r"\bwi-?fi\b"],
    "Drive/VFD/Servo": [r"\bdrive\b", r"\bvfd\b", r"\bsinamics\b", r"\bpowerflex\b", r"\bservo\b", r"\bfrequency\s*converter\b"],
    "Gateway/Protocol Converter": [r"\bgateway\b", r"\bprotocol\s*converter\b", r"\bconverter\b", r"\blink(ing)?\s*device\b", r"\bbridge\b",
                                   r"\bopc\s*gateway\b", r"\bprofi(net|bus)\s*gateway\b"],
    "RTU/Telemetry": [r"\brtu\b", r"\bscadapack\b", r"\btelemetry\b", r"\bremote\s*terminal\s*unit\b"],
    "DCS/Process Control Suite": [r"\bpcs\s*7\b", r"\bcentum\b", r"\bexperion\b", r"\bdelta[vv]?\b", r"\bdcs\b", r"\bsystem\s*800x?a?\b"],
    "Safety Controller/SIS": [r"\bsis\b", r"\btriconex\b", r"\bsafety\s*(controller|instrumented\s*system)\b", r"\bpros?afe\b"],
    "Communication Module": [r"\bsimatic\s*net\s*cp\b", r"\bcp\d{3,}\b", r"\bcommunication\s*processor\b", r"\bcomm\s*module\b",
                             r"\bethernet\s*module\b", r"\bprofinet\s*module\b", r"\bprofibus\s*module\b"],
    "Remote Access/VPN": [r"\be?won\b", r"\bcosy\b", r"\bremote\s*access\b", r"\bvpn\b", r"\bteamviewer\b"],
    "Cloud/Enterprise App": [r"\bcloud\b", r"\bsaas\b", r"\banalytics\b", r"\bportal\b", r"\biiot\b", r"\bedge\s*cloud\b"],
    "Field Device/I-O": [r"\bremote\s*i/?o\b", r"\bio\s*module\b", r"\bfieldbus\b", r"\bsensor\b", r"\bactuator\b", r"\bhart\b",
                         r"\btransmitter\b", r"\bvalve\b", r"\bpower\s*supply\b", r"\bsitop\b"],
}
rx = {k:[re.compile(p, re.IGNORECASE) for p in v] for k,v in patterns.items()}

def classify(text):
    t = str(text or "")
    for asset, regs in rx.items():
        if any(r.search(t) for r in regs):
            return asset
    return "Other/Software"

def zone(asset, text=""):
    pt = (text or "").lower()
    if asset in ["PLC/Controller","Drive/VFD/Servo","Field Device/I-O","Communication Module","RTU/Telemetry","Safety Controller/SIS"]:
        return "Level 0-1 (Field/Basic Control)"
    if asset in ["HMI/SCADA","Gateway/Protocol Converter","DCS/Process Control Suite"]:
        return "Level 2 (Area Supervisory)"
    if asset in ["Engineering Workstation","Historian/Database","OPC Server/Middleware"]:
        return "Level 3 (Operations/Plant Network)"
    if asset in ["Remote Access/VPN"] or "dmz" in pt:
        return "Level 3.5 (Industrial DMZ)"
    if asset in ["Cloud/Enterprise App"]:
        return "Level 4/5 (Enterprise/Cloud)"
    if asset in ["Industrial Network (Switch/Router/Firewall)","Industrial Wireless/AP"]:
        if re.search(r"access\s*point|iwlan|wireless", pt): return "Level 1-2 (Cell/Area Network)"
        if re.search(r"firewall|router|l3|core", pt): return "Level 3 (Operations/Plant Network)"
        return "Conduit (between zones)"
    return "Unknown"

agg["Asset_Type_from_ProductAffected"] = agg["Product_Affected"].apply(classify)
agg["Likely_Zone_from_ProductAffected"] = agg.apply(lambda r: zone(r["Asset_Type_from_ProductAffected"], r["Product_Affected"]), axis=1)

base = pd.read_csv(BASE, dtype=str)
merged = base.merge(agg[["ICS-CERT_Number","Asset_Type_from_ProductAffected","Likely_Zone_from_ProductAffected"]], on="ICS-CERT_Number", how="left")
merged.to_csv(OUT, index=False)
print(f"Wrote {len(merged)} rows to {OUT}")
